In [1]:
import numpy as np
import pandas as pd

In [5]:
np.random.seed(42)

conv_weights = np.random.randn(8, 3, 3, 3).astype(np.float32)

conv_weights[0] *= 0.1
conv_weights[1] *= 0.5
conv_weights[6] *= 5.0
conv_weights[7] *= 10.0

In [7]:
def per_tensor_quantize(tensor):
    max_abs = np.max(np.abs(tensor))

    if max_abs == 0:
        scale = 1.0
    else:
        scale = max_abs / 127

    quantized = np.round(tensor / scale)
    quantized = np.clip(quantized, -127, 127).astype(np.int8)

    dequantized = quantized.astype(np.float32) * scale

    return quantized, dequantized, scale

In [9]:
def per_channel_quantize(tensor):

    quantized = np.zeros_like(tensor, dtype=np.int8)
    dequantized = np.zeros_like(tensor, dtype=np.float32)

    scales = []

    for c in range(tensor.shape[0]):

        channel = tensor[c]

        max_abs = np.max(np.abs(channel))

        if max_abs == 0:
            scale = 1.0
        else:
            scale = max_abs / 127

        q = np.round(channel / scale)
        q = np.clip(q, -127, 127).astype(np.int8)

        dq = q.astype(np.float32) * scale

        quantized[c] = q
        dequantized[c] = dq

        scales.append(scale)

    return quantized, dequantized, np.array(scales)

In [11]:
pt_q, pt_dq, pt_scale = per_tensor_quantize(conv_weights)

pc_q, pc_dq, pc_scales = per_channel_quantize(conv_weights)

In [13]:
rows = []

for c in range(conv_weights.shape[0]):

    original = conv_weights[c]

    pt_mae = np.mean(np.abs(original - pt_dq[c]))

    pc_mae = np.mean(np.abs(original - pc_dq[c]))

    ch_min = np.min(original)
    ch_max = np.max(original)

    better = "Per-Channel" if pc_mae < pt_mae else "Per-Tensor"

    rows.append([
        c,
        f"({ch_min:.3f}, {ch_max:.3f})",
        pt_scale,
        pt_mae,
        pc_scales[c],
        pc_mae,
        better
    ])

In [15]:
avg_pt = np.mean([r[3] for r in rows])
avg_pc = np.mean([r[5] for r in rows])

rows.append([
    "Average",
    "-",
    pt_scale,
    avg_pt,
    "-",
    avg_pc,
    "Per-Channel" if avg_pc < avg_pt else "Per-Tensor"
])

In [17]:
df = pd.DataFrame(
    rows,
    columns=[
        "Channel",
        "Range (min,max)",
        "Per-Tensor Scale",
        "Per-Tensor MAE",
        "Per-Ch Scale",
        "Per-Ch MAE",
        "Better"
    ]
)

print(df)

   Channel    Range (min,max)  Per-Tensor Scale  Per-Tensor MAE Per-Ch Scale  \
0        0    (-0.191, 0.158)          0.303365        0.071464     0.001507   
1        1    (-0.980, 0.926)          0.303365        0.072564     0.007715   
2        2    (-2.620, 1.565)          0.303365        0.072175     0.020628   
3        3    (-1.464, 1.886)          0.303365        0.071594     0.014852   
4        4    (-1.919, 2.463)          0.303365        0.070571     0.019396   
5        5    (-1.607, 1.866)          0.303365        0.068939     0.014691   
6        6   (-5.354, 13.601)          0.303365        0.077800     0.107093   
7        7  (-15.148, 38.527)          0.303365        0.064038     0.303365   
8  Average                  -          0.303365        0.071143            -   

   Per-Ch MAE       Better  
0    0.000326  Per-Channel  
1    0.001661  Per-Channel  
2    0.005373  Per-Channel  
3    0.003731  Per-Channel  
4    0.003995  Per-Channel  
5    0.003901  Per-Channe

# Analysis

Per-tensor quantization uses a single scale for the entire weight tensor. If different output channels have different value ranges, this common scale is influenced by the channel with the largest values. As a result, channels with smaller ranges lose precision because their values are represented using relatively large quantization steps.

Per-channel quantization assigns an independent scale to each output channel. This allows each channel to use a scale that matches its own value distribution, reducing rounding error and improving reconstruction accuracy.

Channels **0** and **1** benefit the most because their weights were intentionally scaled by **0.1** and **0.5**, giving them much smaller value ranges than the other channels.

With per-tensor quantization, these small values are quantized using a scale determined by the largest channel, resulting in poor precision. Per-channel quantization computes much smaller scales for these channels, preserving their values more accurately.


Channels **6** and **7** have the largest value ranges and therefore strongly influence the global scale used in per-tensor quantization.

Since the per-tensor scale is already close to the optimal scale for these channels, using separate scales provides only a small improvement. Consequently, the reconstruction errors for these channels are similar under both methods.

Per-tensor quantization is simpler because only one scale value needs to be stored and used during inference. This reduces implementation complexity and metadata overhead.

Per-channel quantization requires storing one scale for each output channel and slightly increases computation and memory for the scale values. However, it usually provides significantly lower quantization error and better model accuracy, especially for convolution layers with channels that have different value distributions.